# Step 05.5 — What Learning Really Means

这一节不增加新的求导技巧，而是把整个训练系统归位。

目标：区分清楚 **architecture、parameter、data、loss、gradient、optimizer、inference** 各自负责什么。

## 1. “模型学习了”到底是什么意思？

最底层地说：

> **模型的 parameter 被训练数据通过 loss / gradient / optimizer 改变了。**

例如我们一直使用：

```text
prediction = w*x
```

训练前：

```text
w = 3
```

训练后：

```text
w ≈ 5
```

真正改变的是 `w`，而不是公式 `prediction = w*x` 本身。

In [ ]:
import mlx.core as mx

x = mx.array(2.0)
target = mx.array(10.0)

def loss_fn(w):
    prediction = w * x
    return mx.square(prediction - target)

loss_and_grad_fn = mx.value_and_grad(loss_fn)

w = mx.array(3.0)
lr = 0.1

before = w

for _ in range(8):
    loss, grad = loss_and_grad_fn(w)
    w = w - lr * grad
    mx.eval(w)

after = w

print("parameter before:", before)
print("parameter after :", after)
print("architecture    : prediction = w * x")

所以：

```text
architecture 不变
parameter 改变
```

这就是绝大多数常规神经网络训练的基本形式。

当然，研究中也存在 architecture search、dynamic networks 等方法，但那不是我们当前 MiniGPT 的训练方式。

## 2. Parameter、Architecture、Hyperparameter 不要混

以我们后面的 MiniGPT 为例：

### Architecture

决定“网络长什么样”：

```text
几层 Transformer Block
每层有没有 Attention
MLP 在哪里
Residual 怎么连接
```

### Hyperparameter

由我们设定、训练过程不会自动通过普通 gradient descent 学出来：

```text
n_embd = 128
n_head = 4
n_layer = 4
block_size = 64
learning_rate = ...
```

### Parameter

真正被 optimizer 更新的数组：

```text
Linear weights
Embedding weights
RMSNorm scale
...
```

## 3. Data、Loss、Gradient、Optimizer 各负责什么？

把训练系统拆开：

```text
DATA
提供 input 和 target
        ↓
MODEL(parameters)
产生 prediction
        ↓
LOSS FUNCTION
定义什么叫“错”
        ↓
AUTODIFF
计算 gradient
        ↓
OPTIMIZER
根据 gradient 更新 parameters
        ↓
新的 MODEL(parameters)
```

它们不能互相替代：

- 数据决定模型能看到什么经验；
- loss 决定我们奖励/惩罚什么行为；
- gradient 给出局部敏感度；
- optimizer 决定怎样利用 gradient 更新参数；
- architecture 决定模型有哪些计算路径和参数位置。

## 4. Training 和 Inference 是两种不同阶段

### Training

```text
forward
→ loss
→ gradient
→ parameter update
```

parameter 会变化。

### Inference

```text
input
→ forward
→ output
```

正常推理时不需要用 target 算 loss，也不需要 backward/update；parameter 通常保持固定。

以后 MiniGPT 训练和生成文本就是这两个阶段。

## 5. Training loss 下降 ≠ 一定真正学会泛化

如果模型只是把训练样本死记住，也可能让 training loss 很低。

真正希望的是：

> 对没有参与参数更新的新数据，也能做出合理预测。

这叫 **generalization（泛化）**。

因此后面 Tiny Shakespeare 会分：

```text
train data       -> 用来更新 parameter
validation data  -> 不用来训练，只用来检查泛化
```

这也是为什么单看 training loss 不够。

## 把 Step 5 压缩成一张图

```text
             architecture / hyperparameters
                       │
                       ▼
data ──────> model(parameters) ──────> prediction
  │                                      │
  └──────── target ──────────────────────┤
                                         ▼
                                        loss
                                         │
                                      autodiff
                                         │
                                      gradient
                                         │
                                      optimizer
                                         │
                                         ▼
                                 updated parameters
                                         │
                                         └──> 下一轮
```

### Step 5 最终结论

- 学习主要发生在 parameter 数值的更新上；
- architecture 定义计算结构，parameter 填充其中的可学习数值；
- data 提供经验，loss 定义目标；
- autodiff 算 gradient，optimizer 使用 gradient 更新 parameter；
- training 会修改参数，inference 通常只做 forward；
- training loss 低不等于泛化一定好；
- 下一步 Step 6 开始把这些通用学习机制带回语言：**文本怎样变成 token，以及 next-token prediction 到底是什么训练任务。**